In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df=pd.read_csv("../dataset/engineered/zomato_engineered.csv")

In [4]:
df.shape

(41226, 8)

In [5]:
df.head()

,name,cuisines,rest_type,location,rate,votes,approx_cost(for two people),restaurant_profile
0,Jalsa,"north indian, mughlai, chinese",casual dining,banashankari,4.1,775,800.0,"north indian, mughlai, chinese casual dining b..."
1,Spice Elephant,"chinese, north indian, thai",casual dining,banashankari,4.1,787,800.0,"chinese, north indian, thai casual dining bana..."
2,San Churro Cafe,"cafe, mexican, italian","cafe, casual dining",banashankari,3.8,918,800.0,"cafe, mexican, italian cafe, casual dining ban..."
3,Addhuri Udupi Bhojana,"south indian, north indian",quick bites,banashankari,3.7,88,300.0,"south indian, north indian quick bites banasha..."
4,Grand Village,"north indian, rajasthani",casual dining,basavanagudi,3.8,166,600.0,"north indian, rajasthani casual dining basavan..."


In [6]:
df["rate"].dtype

dtype('float64')

In [7]:
df["rate"].describe()

count    41226.000000
mean         3.702091
std          0.440063
min          1.800000
25%          3.400000
50%          3.700000
75%          4.000000
max          4.900000
Name: rate, dtype: float64

In [8]:
df["rating_score"]=df["rate"]/5

In [9]:
df[["name","rate","rating_score"]].head()

,name,rate,rating_score
0,Jalsa,4.1,0.82
1,Spice Elephant,4.1,0.82
2,San Churro Cafe,3.8,0.76
3,Addhuri Udupi Bhojana,3.7,0.74
4,Grand Village,3.8,0.76


In [10]:
df["popularity_score"]=np.log1p(df["votes"])

In [11]:
df["popularity_score"]=(
    df["popularity_score"] - df["popularity_score"].min()
)/(
    df["popularity_score"].max() - df["popularity_score"].min()
)

In [12]:
df["popularity_score"].describe()

count    41226.000000
mean         0.455395
std          0.173670
min          0.000000
25%          0.317646
50%          0.442990
75%          0.578682
max          1.000000
Name: popularity_score, dtype: float64

In [13]:
df["quality_score"]=(
    0.7 * df["rating_score"] + 0.3 * df["popularity_score"]
)

In [14]:
df[
    ["name","rate","votes","rating_score","popularity_score","quality_score"]
].head(10)

,name,rate,votes,rating_score,popularity_score,quality_score
0,Jalsa,4.1,775,0.82,0.683803,0.779141
1,Spice Elephant,4.1,787,0.82,0.685380,0.779614
2,San Churro Cafe,3.8,918,0.76,0.701184,0.742355
3,Addhuri Udupi Bhojana,3.7,88,0.74,0.461267,0.656380
4,Grand Village,3.8,166,0.76,0.525942,0.689783
5,Timepass Dinner,3.8,286,0.76,0.581587,0.706476
6,Rosewood International Hotel - Bar & Restaurant,3.6,8,0.72,0.225794,0.571738
7,Onesta,4.6,2556,0.92,0.806342,0.885903
8,Penthouse Cafe,4.0,324,0.80,0.594365,0.738310
9,Smacznego,4.2,504,0.84,0.639656,0.779897


In [15]:
tfidf=TfidfVectorizer(stop_words="english")
tfidf_matrix=tfidf.fit_transform(
    df["restaurant_profile"]
)

In [16]:
tfidf_matrix.shape

(41226, 244)

In [17]:
def hybrid_recommend_restaurants(restaurant_name, top_n=10):
    if restaurant_name not in df["name"].values:
        return f"Restaurant '{restaurant_name}' not found"
    idx=df[df["name"]==restaurant_name].index[0]
    similarity_scores=cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()
    hybrid_scores=(
        0.8*similarity_scores+0.2*df["quality_score"].values
    )
    similar_indices=hybrid_scores.argsort()[::-1]
    similar_indices=similar_indices[
        similar_indices!=idx
    ]
    top_indices=similar_indices[:top_n]
    recommendations=df.iloc[top_indices][
        [
            "name",
            "cuisines",
            "location",
            "rate",
            "votes"
        ]
    ].copy()
    recommendations["similarity_score"]=(
        similarity_scores[top_indices]
    )
    recommendations["quality_score"]=(
        df["quality_score"].iloc[top_indices].values
    )
    recommendations["hybrid_score"]=(
        hybrid_scores[top_indices]
    )
    recommendations=recommendations.sort_values(
        by="hybrid_score",
        ascending=False
    )
    return recommendations.reset_index(drop=True)

In [18]:
hybrid_recommend_restaurants("Jalsa")

,name,cuisines,location,rate,votes,similarity_score,quality_score,hybrid_score
0,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,804,1.000000,0.780272,0.956054
1,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,783,1.000000,0.779457,0.955891
2,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,783,1.000000,0.779457,0.955891
3,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,783,1.000000,0.779457,0.955891
4,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,775,1.000000,0.779141,0.955828
5,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,775,1.000000,0.779141,0.955828
6,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,775,1.000000,0.779141,0.955828
7,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,775,1.000000,0.779141,0.955828
8,Food Box Cafe,"mughlai, north indian, chinese",banashankari,3.6,36,1.000000,0.615321,0.923064
9,1947,"north indian, chinese",banashankari,4.0,808,0.839312,0.766425,0.824735


In [ ]:
hybrid_recommend_restaurants("Jalsa", top_n=10)

,name,cuisines,location,rate,votes,similarity_score,quality_score,hybrid_score
0,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,804,1.000000,0.780272,0.956054
1,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,783,1.000000,0.779457,0.955891
2,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,783,1.000000,0.779457,0.955891
3,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,783,1.000000,0.779457,0.955891
4,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,775,1.000000,0.779141,0.955828
5,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,775,1.000000,0.779141,0.955828
6,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,775,1.000000,0.779141,0.955828
7,Jalsa,"north indian, mughlai, chinese",banashankari,4.1,775,1.000000,0.779141,0.955828
8,Food Box Cafe,"mughlai, north indian, chinese",banashankari,3.6,36,1.000000,0.615321,0.923064
9,1947,"north indian, chinese",banashankari,4.0,808,0.839312,0.766425,0.824735
